在多头注意力中，给定相同的查询、键和值的集合，模型可以学习到不同的行为，并将这些行为组合起来以捕获序列内各种范围的依赖关系。为此，它通过独立学习得到的$h$组不同的线性投影来变换查询、键和值。

这些$h$组变换后的查询、键和值会并行地送到注意力汇聚中。然后，将这$h$个注意力汇聚的输出拼接在一起，并通过另一个可学习的线性投影进行变换，以产生最终输出。每个注意力汇聚的输出被称为一个“头”。

### 模型

形式上，给定查询$\mathbf{q} \in \mathbb{R}^{d_q}$、键$\mathbf{k} \in \mathbb{R}^{d_k}$和值$\mathbf{v} \in \mathbb{R}^{d_v}$，每个注意力头$\mathbf{h}_i$（$i = 1, \ldots, h$）的计算方法如下：

$$\mathbf{h}_i = f(\mathbf W_i^{(q)}\mathbf q, \mathbf W_i^{(k)}\mathbf k,\mathbf W_i^{(v)}\mathbf v) \in \mathbb R^{p_v},$$

其中，可学习的参数包括$\mathbf W_i^{(q)}\in\mathbb R^{p_q\times d_q}$、$\mathbf W_i^{(k)}\in\mathbb R^{p_k\times d_k}$和$\mathbf W_i^{(v)}\in\mathbb R^{p_v\times d_v}$，函数$f$代表注意力汇聚。

多头注意力的最终输出通过另一个线性转换得到，其可学习参数是$\mathbf W_o\in\mathbb R^{p_o\times h p_v}$：

$$\mathbf W_o \begin{bmatrix}\mathbf h_1\\\vdots\\\mathbf h_h\end{bmatrix} \in \mathbb{R}^{p_o}.$$

这种设计使得每个头都可能关注输入的不同部分，从而能够表示比简单加权平均值更复杂的函数。

In [1]:
import math
import torch
from torch import nn

实现

选择缩放点积注意力作为每一个注意力头

为了避免计算代价和参数代价的大幅增长，
我们设定$p_q = p_k = p_v = p_o / h$。
值得注意的是，如果将查询、键和值的线性变换的输出数量设置为
$p_q h = p_k h = p_v h = p_o$，
则可以并行计算$h$个头。

In [2]:
import math
import torch
from torch import nn
import torch.nn.functional as F

def masked_softmax(X, valid_lens):
    """Perform softmax operation by masking elements on the last axis."""
    if valid_lens is None:
        return F.softmax(X, dim=-1)
    else:
        shape = X.shape
        if valid_lens.dim() == 1:
            mask = torch.arange(shape[-1], dtype=torch.float32, device=X.device)[None, :] >= valid_lens[:, None]
            X = X.masked_fill(mask.unsqueeze(1), -1e9)
        elif valid_lens.dim() == 2:
            mask = torch.arange(shape[-1], dtype=torch.float32, device=X.device)[None, None, :] >= valid_lens[:, :, None]
            X = X.masked_fill(mask, -1e9)
        return F.softmax(X, dim=-1)

def transpose_qkv_torch(X, num_heads):
    """Transposition for parallel computation of multiple attention heads."""
    # Input X shape: (batch_size, seq_len, num_hiddens)
    # Output X shape: (batch_size, seq_len, num_heads, num_hiddens // num_heads)
    X = X.reshape(X.shape[0], X.shape[1], num_heads, -1)

    # Output X shape: (batch_size, num_heads, seq_len, num_hiddens // num_heads)
    X = X.permute(0, 2, 1, 3)

    # Output X shape: (batch_size * num_heads, seq_len, num_hiddens // num_heads)
    return X.reshape(-1, X.shape[2], X.shape[3])

def transpose_output_torch(X, num_heads):
    """Reverse the transposition operation for multi-head attention."""
    # Input X shape: (batch_size * num_heads, seq_len, num_hiddens // num_heads)

    batch_size = X.shape[0] // num_heads

    # Output X shape: (batch_size, num_heads, seq_len, num_hiddens // num_heads)
    X = X.reshape(batch_size, num_heads, X.shape[1], X.shape[2])

    # Output X shape: (batch_size, seq_len, num_heads, num_hiddens // num_heads)
    X = X.permute(0, 2, 1, 3)

    # Output X shape: (batch_size, seq_len, num_hiddens)
    return X.reshape(batch_size, X.shape[1], -1)

class DotProductAttention(nn.Module):
    """Scaled dot product attention with optional dropout and masking."""
    def __init__(self, dropout, **kwargs):
        super(DotProductAttention, self).__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, keys, values, valid_lens=None):
        d = queries.shape[-1]
        scores = torch.bmm(queries, keys.transpose(1, 2)) / math.sqrt(d)
        self.attention_weights = masked_softmax(scores, valid_lens)
        return torch.bmm(self.dropout(self.attention_weights), values)

class MultiHeadAttention(nn.Module):
    """多头注意力"""
    def __init__(self, key_size, query_size, value_size, num_hiddens,
                 num_heads, dropout, bias=False, **kwargs):
        super(MultiHeadAttention, self).__init__(**kwargs)
        self.num_heads = num_heads
        # Use our custom DotProductAttention
        self.attention = DotProductAttention(dropout)
        self.W_q = nn.Linear(query_size, num_hiddens, bias=bias)
        self.W_k = nn.Linear(key_size, num_hiddens, bias=bias)
        self.W_v = nn.Linear(value_size, num_hiddens, bias=bias)
        self.W_o = nn.Linear(num_hiddens, num_hiddens, bias=bias)

    def forward(self, queries, keys, values, valid_lens):
        # Apply linear projections for queries, keys, and values
        queries_proj = self.W_q(queries)
        keys_proj = self.W_k(keys)
        values_proj = self.W_v(values)

        # Transpose for multi-head attention
        queries_transformed = transpose_qkv_torch(queries_proj, self.num_heads)
        keys_transformed = transpose_qkv_torch(keys_proj, self.num_heads)
        values_transformed = transpose_qkv_torch(values_proj, self.num_heads)

        if valid_lens is not None:
            if valid_lens.dim() == 1:
                valid_lens = torch.repeat_interleave(valid_lens, repeats=self.num_heads, dim=0)
            else:
                valid_lens = valid_lens.repeat_interleave(self.num_heads, dim=0)

        # Apply scaled dot product attention
        output = self.attention(queries_transformed, keys_transformed, values_transformed, valid_lens)

        # Concat and apply final linear layer
        output_concat = transpose_output_torch(output, self.num_heads)
        return self.W_o(output_concat)

为了能多头并行计算，MultiHeadAttention类将使用下面定义的两个转置函数。

In [6]:
def transpose_qkv(X, num_heads):
    """为了多注意力头的并行计算而变换形状"""
    # 输入X的形状:(batch_size，查询或者“键－值”对的个数，num_hiddens)
    # 输出X的形状:(batch_size，查询或者“键－值”对的个数，num_heads，
    # num_hiddens/num_heads)
    X = X.reshape(X.shape[0], X.shape[1], num_heads, -1)

    # 输出X的形状:(batch_size，num_heads，查询或者“键－值”对的个数,
    # num_hiddens/num_heads)
    X = X.permute(0, 2, 1, 3)

    # 最终输出的形状:(batch_size*num_heads,查询或者“键－值”对的个数,
    # num_hiddens/num_heads)
    return X.reshape(-1, X.shape[2], X.shape[3])


def transpose_output(X, num_heads):
    """逆转transpose_qkv函数的操作"""
    X = X.reshape(-1, num_heads, X.shape[1], X.shape[2])
    X = X.permute(0, 2, 1, 3)
    return X.reshape(X.shape[0], X.shape[1], -1)

测试这个类

In [4]:
num_hiddens, num_heads = 100, 5
attention = MultiHeadAttention(num_hiddens, num_hiddens, num_hiddens,
                               num_hiddens, num_heads, 0.5)
attention.eval()

MultiHeadAttention(
  (attention): DotProductAttention(
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (W_q): Linear(in_features=100, out_features=100, bias=False)
  (W_k): Linear(in_features=100, out_features=100, bias=False)
  (W_v): Linear(in_features=100, out_features=100, bias=False)
  (W_o): Linear(in_features=100, out_features=100, bias=False)
)

In [5]:
batch_size, num_queries = 2, 4
num_kvpairs, valid_lens =  6, torch.tensor([3, 2])
X = torch.ones((batch_size, num_queries, num_hiddens))
Y = torch.ones((batch_size, num_kvpairs, num_hiddens))
attention(X, Y, Y, valid_lens).shape

torch.Size([2, 4, 100])